<a href="https://colab.research.google.com/github/DanielYaari28/AIPI510P1/blob/daniel-feature-engineering/notebooks/01_data_cleaning_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Source

The raw dataset used in this notebook is the U.S. Department of Education
College Scorecard's "Most Recent Institution-Level Data."

The raw dataset is not stored in this repository because of its file size.
It can be downloaded from:

https://collegescorecard.ed.gov/data/

After downloading, upload `Most-Recent-Cohorts-Institution.csv` to the
Colab session before running this notebook.

In [106]:
import pandas as pd

file_path = "/content/Most-Recent-Cohorts-Institution.csv"

df = pd.read_csv(
    file_path,
    low_memory=False,
    on_bad_lines="skip"
)

print(df.shape)
df.head()

(5655, 3308)


,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11,SCORECARD_SECTOR,EARN_THR_STATE,EARN_THR_NAT
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0,4.0,32204.0,36082.0
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0,4.0,32204.0,36082.0
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0,5.0,32204.0,36082.0
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,uah.clearcostcalculator.com/student/default/ne...,...,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0,4.0,32204.0,36082.0
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,tcc.ruffalonl.com/Alabama State University/Fre...,...,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0,4.0,32204.0,36082.0


In [107]:
print(df["PREDDEG"].dtype)
print(df["UGDS"].dtype)

print(df["PREDDEG"].head())
print(df["UGDS"].head())

object
float64
0    3
1    3
2    3
3    3
4    3
Name: PREDDEG, dtype: object
0     6124.0
1    11635.0
2      241.0
3     6591.0
4     3477.0
Name: UGDS, dtype: float64


In [108]:
df["PREDDEG"] = pd.to_numeric(
    df["PREDDEG"],
    errors="coerce"
)

In [109]:
#Institutions considerations:
eligible = df[
    (df["PREDDEG"] == 3) & #undergrad
    (df["UGDS"] >= 2000) #above 2,000 undergrads
].copy()

print("Eligible universities:", len(eligible))

Eligible universities: 691


In [110]:
project_cols = [
    "UNITID",
    "INSTNM",
    "STABBR",
    "CONTROL",
    "UGDS",

    # Student Success
    "C150_4",
    "RET_FT4",

    # Affordability
    "NPT4_PUB",
    "NPT4_PRIV",
    "DEBT_MDN",
    "COSTT4_A",

    # Outcomes
    "MD_EARN_WNE_P10",

    # Accessibility
    "PCTPELL",

    # Context
    "ADM_RATE"
]

project_df = eligible[project_cols].copy()

In [111]:
project_df.info()

print("\nMissing values:")
print(project_df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 691 entries, 0 to 5022
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   UNITID           691 non-null    int64  
 1   INSTNM           691 non-null    object 
 2   STABBR           691 non-null    object 
 3   CONTROL          691 non-null    object 
 4   UGDS             691 non-null    float64
 5   C150_4           684 non-null    object 
 6   RET_FT4          683 non-null    object 
 7   NPT4_PUB         398 non-null    object 
 8   NPT4_PRIV        281 non-null    object 
 9   DEBT_MDN         687 non-null    object 
 10  COSTT4_A         675 non-null    object 
 11  MD_EARN_WNE_P10  684 non-null    float64
 12  PCTPELL          689 non-null    object 
 13  ADM_RATE         635 non-null    object 
dtypes: float64(2), int64(1), object(11)
memory usage: 81.0+ KB

Missing values:
UNITID               0
INSTNM               0
STABBR               0
CONTROL      

In [112]:
project_df["NET_PRICE"] = (
    project_df["NPT4_PUB"]
    .fillna(project_df["NPT4_PRIV"])
)

In [113]:
print("Missing net price:", project_df["NET_PRICE"].isna().sum())

print(
    "Net price coverage:",
    round(project_df["NET_PRICE"].notna().mean() * 100, 1),
    "%"
)

Missing net price: 12
Net price coverage: 98.3 %


In [114]:
numeric_cols = [
    "C150_4",
    "RET_FT4",
    "NET_PRICE",
    "DEBT_MDN",
    "COSTT4_A",
    "MD_EARN_WNE_P10",
    "PCTPELL",
    "ADM_RATE"
]

project_df[numeric_cols].describe()

,MD_EARN_WNE_P10
count,684.000000
mean,59746.814327
std,16376.212764
min,24328.000000
25%,49129.250000
50%,57163.500000
75%,67580.500000
max,143372.000000


In [115]:
project_df[project_df["DEBT_MDN"].isna()][["INSTNM", "DEBT_MDN"]]

,INSTNM,DEBT_MDN
703,United States Naval Academy,NaN
1543,United States Military Academy,NaN
2024,Grove City College,NaN
4639,University of the People,NaN


In [116]:
project_df[project_df["DEBT_MDN"].isna()][["INSTNM",
"C150_4",
"RET_FT4",
"MD_EARN_WNE_P10",
"PCTPELL"]]

,INSTNM,C150_4,RET_FT4,MD_EARN_WNE_P10,PCTPELL
703,United States Naval Academy,0.9294,0.9732,NaN,NaN
1543,United States Military Academy,0.8706,0.9463,NaN,NaN
2024,Grove City College,0.8317,0.8833,NaN,0
4639,University of the People,0.3354,NaN,NaN,0


In [117]:
ranking_vars = [
    "C150_4",
    "RET_FT4",
    "NET_PRICE",
    "DEBT_MDN",
    "MD_EARN_WNE_P10",
    "PCTPELL"
]

In [118]:
missing_data = project_df[ranking_vars].isna().any(axis=1)
project_df[missing_data][["INSTNM"] + ranking_vars]

,INSTNM,C150_4,RET_FT4,NET_PRICE,DEBT_MDN,MD_EARN_WNE_P10,PCTPELL
7,Athens State University,NaN,NaN,NaN,14861,50273.0,0.4251
703,United States Naval Academy,0.9294,0.9732,NaN,NaN,NaN,NaN
1202,Beth Medrash Govoha,NaN,NaN,NaN,5500,47544.0,0.7449
1268,Thomas Edison State University,NaN,NaN,NaN,7813,69331.0,0.2367
1363,CUNY Graduate School and University Center,NaN,NaN,NaN,9800,65991.0,0.3846
1537,Excelsior University,NaN,NaN,NaN,10870,78493.0,0.2824
1543,United States Military Academy,0.8706,0.9463,NaN,NaN,NaN,NaN
2024,Grove City College,0.8317,0.8833,NaN,NaN,NaN,0
3857,Strayer University-North Carolina,0.3333,NaN,NaN,14000,40092.0,0.8229
4053,Strayer University-Texas,0.25,1,NaN,14000,40092.0,0.7927


In [119]:
complete_df = project_df[~missing_data].copy()

print(len(complete_df))

674


In [120]:
ps = (complete_df["DEBT_MDN"] == "PS")
complete_df[ps][["INSTNM", "DEBT_MDN"]]


,INSTNM,DEBT_MDN
1541,United Talmudical Seminary,PS
3683,Uta Mesivta of Kiryas Joel,PS


In [121]:
from numpy import nan
complete_df["DEBT_MDN"] = complete_df["DEBT_MDN"].replace("PS", nan)

In [122]:
complete_df["DEBT_MDN"] = pd.to_numeric(complete_df["DEBT_MDN"])

In [123]:
complete_df = complete_df[
    ~complete_df[ranking_vars].isna().any(axis=1)
].copy()

In [124]:
print(len(complete_df))
print(complete_df[ranking_vars].isna().sum())

672
C150_4             0
RET_FT4            0
NET_PRICE          0
DEBT_MDN           0
MD_EARN_WNE_P10    0
PCTPELL            0
dtype: int64


In [125]:
complete_df[ranking_vars].describe()

,DEBT_MDN,MD_EARN_WNE_P10
count,672.000000,672.000000
mean,15903.053571,59849.065476
std,4118.010948,16350.719590
min,4500.000000,24328.000000
25%,13000.000000,49363.250000
50%,15621.000000,57163.500000
75%,19000.000000,67580.500000
max,27000.000000,143372.000000


In [126]:
zero = complete_df["C150_4"] == 0
complete_df[zero][["INSTNM", "C150_4"]]

,INSTNM,C150_4


In [127]:
max_Pell = complete_df["PCTPELL"].max()
complete_df[complete_df["PCTPELL"] == max_Pell][["INSTNM", "PCTPELL"]]

,INSTNM,PCTPELL
2827,Universidad Ana G. Mendez-Cupey Campus,0.9928


In [128]:
complete_df[complete_df["UNITID"].duplicated()][["UNITID", "INSTNM"]]

,UNITID,INSTNM


In [129]:
complete_df.to_csv("cleaned_college_data_new.csv", index=False)